In [1]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import pytz

In [2]:
from utils.trends_slope import check_trend_of_price
from collections import Counter

In [3]:
from API.get_data.api_yahoo import get_historical_data, get_analyst_price_targets, get_business_recommendation, get_business_grades
from utils.old_trends import get_trends_events

In [4]:
from utils.trends import get_trends_events
from utils.old_slope import qualify_slope

In [5]:
df = get_historical_data("MSFT", "5y")

In [6]:
df2=df.copy()

In [7]:
today = datetime.now()#.strftime('%Y-%m-%d')
first_of_this_month = today.replace(day=1)
first_of_last_month = first_of_this_month - relativedelta(months=1)

# Remettre l'heure à zéro (minuit)
first_of_this_month_midnight = first_of_this_month.replace(hour=0, minute=0, second=0, microsecond=0)
first_of_last_month_midnight = first_of_last_month.replace(hour=0, minute=0, second=0, microsecond=0)

one_year_rolling = today - relativedelta(months=12)
one_year_rolling_midnight = one_year_rolling.replace(hour=0, minute=0, second=0, microsecond=0)

five_year_rolling = today - relativedelta(year=5)
five_year_rolling_midnight = five_year_rolling.replace(hour=0, minute=0, second=0, microsecond=0)

year_to_date_midnight = datetime(first_of_this_month.year, 1, 1, 0, 0)

# Reset date to LocalTime 
df["Date"] = df["Date"].apply(lambda x: x.tz_localize(None))

# filter Dataframe
this_month_data = df[df["Date"]>=first_of_this_month_midnight]
last_month_data = df[df["Date"]>=first_of_last_month_midnight]
year_to_date_data = df[df["Date"]>=year_to_date_midnight]
one_year_rolling_period_data = df[df["Date"]>=one_year_rolling_midnight]
five_year_rolling_period_data = df[df["Date"]>=five_year_rolling_midnight]

tendance sur  
- mois   
- year to date   
- N-1  
- 5 ans  
- 10 ans   

In [8]:
different_dataframe_filter_bydate = [this_month_data, last_month_data, year_to_date_data, one_year_rolling_period_data, five_year_rolling_period_data]

In [9]:
def check_most_present_value_df(df_name):
    value_of_direction = df_name["EMA_Direction_Slope"].values
    most_present_direction = Counter(value_of_direction).most_common(1)[0][0]
    return most_present_direction

In [23]:
def extract_trend(df_name, col_name: str, degre: int) -> str:
    """
    Computes and qualifies the slope trend of a given column in the DataFrame.

    Args:
        df_name (pd.DataFrame): Filtered DataFrame to analyze.
        col (str): Column name to compute the trend on.
        degre (int): Degree for the polynomial fit. Default is 3.

    Returns:
        str: The trend label : Very bullish, Bullish, Stagnant, Bearish, Very bearish
    """
    slope = get_trends_events(df=df_name, cols=[col_name], degre=degre)
    slope_value = slope[f"{col_name}_slope"]
    return qualify_slope(slope_value)["steepness"]


In [28]:
def today_analyse_price(df):
    return_df = df[["Date", "Close", "Volume"]].tail(1)
    different_dataframe_filter_bydate = [this_month_data, last_month_data, year_to_date_data, one_year_rolling_period_data, five_year_rolling_period_data]
    all_trends = []
    all_volume = []
    for df_filter in different_dataframe_filter_bydate:
        check_trend_of_price(df_filter)
        trend_price = extract_trend(df_name=df_filter, col_name="Close", degre=3)
        all_trends.append(trend_price)
        trend_volume = extract_trend(df_name=df_filter, col_name="Volume", degre=3)
        all_volume.append(trend_volume)
        # slope_price = get_trends_events(
        #     df=df_filter,
        #     cols=["Close"],
        #     degre=3
        # )
        # trend = qualify_slope(slope_price["Close_slope"])["steepness"]
        # all_trends.append(trend)

        # slope_volume = get_trends_events(
        #     df=df_filter,
        #     cols=["Volume"],
        #     degre=3
        # )
        # trend = qualify_slope(slope_volume["Volume_slope"])["steepness"]
        # all_volume.append(trend)

    # allocate the most sxitch there have been during an period (up, down, flat)
    return_df["this_month_most_direction"] = check_most_present_value_df(this_month_data)
    return_df["last_month_most_direction"] = check_most_present_value_df(last_month_data)
    return_df["year_to_date_direction"] = check_most_present_value_df(year_to_date_data)
    return_df["one_year_rolling_period_most_direction"] = check_most_present_value_df(one_year_rolling_period_data)
    return_df["five_year_rolling_period_most_direction"] = check_most_present_value_df(five_year_rolling_period_data)

    # Allocate all trends calculate
    return_df["this_month_trend"] = all_trends[0]
    return_df["last_month_trend"] = all_trends[1]
    return_df["year_to_date_trend"] = all_trends[2]
    return_df["one_year_rolling_trend"] = all_trends[3]
    return_df["five_year_rollingtrend"] = all_trends[4]


    return_df["this_month_volume"] = all_volume[0]
    return_df["last_month_volume"] = all_volume[1]
    return_df["year_to_date_volume"] = all_volume[2]
    return_df["one_year_rolling_volume"] = all_volume[3]
    return_df["five_year_rolling_volume"] = all_volume[4]
    return return_df

In [29]:
analyse = today_analyse_price(df)
analyse.T

c:\Users\cleme\Documents\Ynov\M2\Projet Master\Projet Bourse\NEW\Projet_master\SRC\utils\trends_slope.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_name["EMA"] = df_name["Close"].ewm(span=ema_period, adjust=False).mean()
c:\Users\cleme\Documents\Ynov\M2\Projet Master\Projet Bourse\NEW\Projet_master\SRC\utils\trends_slope.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_name["EMA_Slope"] = (
c:\Users\cleme\Documents\Ynov\M2\Projet Master\Projet Bourse\NEW\Projet_master\SRC\utils\trends_slope

,1255
Date,2025-07-07 00:00:00
Close,497.51001
Volume,3330446
this_month_most_direction,Up
last_month_most_direction,Up
year_to_date_direction,Up
one_year_rolling_period_most_direction,Up
five_year_rolling_period_most_direction,Up
this_month_trend,Very bullish
last_month_trend,Very bullish


---

In [ ]:
# fonction a rajouter a get info company preporcessing -- info business

In [ ]:
import numpy as np
def u(df_trend):
    df_trend["sell"] = df_trend["sell"] / total_recommendation
    df_trend["strongBuy"] = df_trend["strongBuy"] / total_recommendation
    df_trend["strongSell"] = df_trend["strongSell"] / total_recommendation
    mean_sell = np.round(df_trend["sell"].mean(),2)
    mean_strongBuy = np.round(df_trend["strongBuy"].mean(),2)
    mean_strongSell = np.round(df_trend["strongSell"].mean(),2)

In [ ]:
get_business_recommendation("TSLA")

In [ ]:
get_business_grades("AAPL")

In [ ]:
get_analyst_price_targets("AAPL")